In [0]:
# Läs in rensade datan från silver layer
df_silver = spark.read.table("marathos.silver.cleaned_marathon")

# Skapar dim event
dim_event = (
    df_silver.select(
    "event_id",
    "Event name",
    "Year of event",
    "Event dates",
    "Event distance/length",
    "Event number of finishers"
)
.dropDuplicates(["event_id"])
)

dim_event.write.format("delta") \
    .option("delta.columnMapping.mode", "name") \
    .mode("overwrite") \
    .saveAsTable("marathos.gold.dim_event")

# skapa dim atholete
dim_athlete = (
    df_silver.select(
    "athlete_id",
    "Athlete gender",
    "Athlete year of birth",
    "Athlete country",
    "Athlete club",
    "Athlete age category"
)
.dropDuplicates(["athlete_id"])
)

dim_athlete.write.format("delta") \
    .option("delta.columnMapping.mode", "name") \
    .mode("overwrite") \
    .saveAsTable("marathos.gold.dim_athlete")

print("Dimensiontabellerna dim_event och dim_athlete har skapats i marathos.gold")



In [0]:
from pyspark.sql.functions import monotonically_increasing_id

# Skapa fct table med unikt result id
fct_result = (
    df_silver.select(
    "event_id",
    "athlete_id",
    "Athlete performance",
    "Athlete average speed",
    "ingestion_time"
    )
    .withColumn("result_id", monotonically_increasing_id() + 1)
)

fct_result.write.format("delta") \
    .option("delta.columnMapping.mode", "name") \
    .mode("overwrite") \
    .saveAsTable("marathos.gold.fct_result")

print("Faktatabellen fct_results har skapats i marathos.gold")

In [0]:
# Skapar olika vyer för distanslopp

spark.sql("""
CREATE OR REPLACE VIEW marathos.gold.vw_distance_50km AS
SELECT *
FROM marathos.silver.cleaned_marathon
WHERE `Event distance/length` = '50 km'
